# NB 2.3 &mdash; Regressió polinòmica: quan el model aprèn massa

**MP 5134** &mdash; UT2

*Dades: AEMET, estació de l'aeroport de Palma.*

---
### Què farem avui

Al final del NB 2.2 vam topar amb un límit: la temperatura al llarg de l'any
dibuixa una corba, i una recta no la pot seguir. Avui aprendrem a **doblegar la
recta**.

Però aquest notebook no va realment de corbes. Va del problema més important de
tot l'aprenentatge automàtic, i el veurem amb els nostres ulls: **un model pot
aprendre tan bé les dades d'entrenament que deixi de servir per a dades noves.**

Quan acabis hauries de saber respondre aquestes preguntes:

- Com pot una regressió *lineal* dibuixar una corba?
- Què és el sobreajust i què és l'infraajust?
- Com es detecta el sobreajust, si no el podem dibuixar?
- Què podem fer quan un model sobreajusta?

## 1. Les dades, amb el segell posat

Comencem, com hem promès, amb les línies que protegeixen el test segellat. Després
fem la mateixa partició d'entrenament i validació del NB 2.2.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL_DADES = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/disseny-avaluacio-models-ml/refs/heads/main/UT01-Entorn_de_treball_primer_model/aemet/meteo_palma.csv"
df = pd.read_csv(URL_DADES, parse_dates=["fecha"])

# El test segellat (2024-2025) no es toca fins a la UT10
TALL_SEGELLAT = "2024-01-01"
dades = df[df["fecha"] < TALL_SEGELLAT].dropna(subset=["tmed"]).copy()

TALL_VALIDACIO = "2022-01-01"
train = dades[dades["fecha"] < TALL_VALIDACIO].copy()
valid = dades[dades["fecha"] >= TALL_VALIDACIO].copy()

print(f"Dies d'entrenament (2015-2021): {len(train)}")
print(f"Dies de validació  (2022-2023): {len(valid)}")

Avui volem predir una cosa més senzilla que al NB 2.2: la **temperatura mitjana
d'un dia** (`tmed`) a partir d'una sola dada, **en quin moment de l'any som**.
No és un model gaire útil a la vida real, però té una virtut enorme per aprendre:
com que només té una entrada, tot el que faci el model es pot dibuixar.

## 2. La temperatura al llarg de l'any

In [ ]:
plt.figure(figsize=(8, 4))
plt.scatter(train["dia_any"], train["tmed"], alpha=0.15, s=8)
plt.xlabel("Dia de l'any (1 = 1 de gener)")
plt.ylabel("Temperatura mitjana (°C)")
plt.title("Set anys de temperatures, dia a dia")
plt.show()

Cada punt és un dia de 2015 a 2021. Es veu una onada molt clara: uns 10 °C de
mitjana al gener, uns 26 °C al juliol i a l'agost, i de tornada al fred al desembre. Al
voltant de l'onada hi ha gruix, perquè no tots els 15 d'agost fan la mateixa
calor.

## 3. Una recta no pot fer-ho

Abans de res, preparem una columna nova: **la fracció de l'any**, un número que va
de 0 (inici de l'any) a 1 (final de l'any). És el mateix que `dia_any`, però en
una escala més petita. De seguida veuràs per què ens convé.

In [ ]:
for taula in (train, valid):
    taula["fraccio_any"] = taula["dia_any"] / 366

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

recta = LinearRegression().fit(train[["fraccio_any"]], train["tmed"])

any_sencer = pd.DataFrame({"fraccio_any": np.linspace(0, 1, 366)})
dies_grafica = any_sencer["fraccio_any"] * 366

plt.figure(figsize=(8, 4))
plt.scatter(train["dia_any"], train["tmed"], alpha=0.15, s=8)
plt.plot(dies_grafica, recta.predict(any_sencer), color="tab:red", linewidth=2)
plt.xlabel("Dia de l'any")
plt.ylabel("Temperatura mitjana (°C)")
plt.show()

print(f"MAE de validació de la recta: {mean_absolute_error(valid['tmed'], recta.predict(valid[['fraccio_any']])):.2f} °C")

La recta ho intenta, però no té cap manera de fer una muntanya: s'equivoca uns
**5 °C** de mitjana.

Aquest problema té nom. Quan un model és **massa simple per a la forma que tenen les
dades**, diem que fa **infraajust** (*underfitting*). L'infraajust es reconeix
perquè el model s'equivoca molt **fins i tot amb les dades d'entrenament**: no és
que no sàpiga generalitzar, és que no ha arribat a aprendre.

## 4. Com doblegar una recta

Aquí ve el truc, i és molt més senzill del que sembla.

Una regressió lineal **només sap sumar columnes multiplicades per un número**. No
la podem canviar. El que sí podem fer és **donar-li columnes noves**, fabricades a
partir de la que ja tenim: la fracció de l'any multiplicada per ella mateixa, una
altra vegada per ella mateixa, i així successivament.

Scikit-learn té una eina que fabrica aquestes columnes: `PolynomialFeatures`.
Mirem què fa amb tres dies d'exemple.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

exemple = pd.DataFrame({"fraccio_any": [0.1, 0.5, 0.9]})

fabrica = PolynomialFeatures(degree=3, include_bias=False)
columnes_noves = fabrica.fit_transform(exemple)

pd.DataFrame(columnes_noves, columns=fabrica.get_feature_names_out())

D'**una** columna n'hem fet **tres**:

- `fraccio_any`, la original.
- `fraccio_any^2`, la mateixa multiplicada per ella mateixa (0,5 × 0,5 = 0,25).
- `fraccio_any^3`, multiplicada tres vegades (0,5 × 0,5 × 0,5 = 0,125).

Amb aquestes tres columnes, el model continua fent el mateix de sempre: sumar-les
multiplicades per un número. **Però el resultat ja no és una recta**, perquè
aquestes columnes creixen de maneres diferents i, combinades, poden fer pujades i
baixades. No cal que sàpigues per què funciona; el que importa és veure-ho.

El número `degree` és el **grau**: fins a quantes multiplicacions arribem. Grau 1
és la recta de sempre. Grau 2 permet una corba amb una sola pujada o baixada.
Com més grau, més revolts pot fer la corba.

Ara ja pots entendre per què hem fet servir la fracció de l'any i no el dia. El
dia 366 multiplicat quinze vegades per ell mateix dóna un número de quaranta
xifres, i els ordinadors perden precisió amb números així. La fracció de l'any
és sempre com a molt 1, i 1 per 1 continua sent 1.

`include_bias=False` evita que la fàbrica afegeixi una columna plena d'uns. No la
necessitem, perquè `LinearRegression` ja té la seva ordenada a l'origen.

Recorda la taula de la UT1: **el grau és un hiperparàmetre**. El model no el tria;
el triem nosaltres. I tota la resta del notebook va de com triar-lo bé.

### 4.1 Corbes de grau 2 i de grau 4

Preparem tres funcions petites: una que fabrica les columnes i entrena el model,
una que fa prediccions i una que en calcula l'error. Les farem servir moltes
vegades.

In [ ]:
def entrenar_polinomi(grau, dades_entrenament):
    """Fabrica les columnes polinòmiques i entrena una regressió lineal amb elles."""
    fabrica = PolynomialFeatures(degree=grau, include_bias=False)
    X = fabrica.fit_transform(dades_entrenament[["fraccio_any"]])
    model = LinearRegression().fit(X, dades_entrenament["tmed"])
    return fabrica, model


def predir_polinomi(fabrica, model, taula):
    return model.predict(fabrica.transform(taula[["fraccio_any"]]))


def mae_polinomi(fabrica, model, taula):
    return mean_absolute_error(taula["tmed"], predir_polinomi(fabrica, model, taula))

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for eix, grau in zip(eixos, [2, 4]):
    fabrica, model = entrenar_polinomi(grau, train)
    eix.scatter(train["dia_any"], train["tmed"], alpha=0.15, s=8)
    eix.plot(dies_grafica, predir_polinomi(fabrica, model, any_sencer), color="tab:red", linewidth=2)
    eix.set_title(f"Grau {grau}: MAE de validació {mae_polinomi(fabrica, model, valid):.2f} °C")
    eix.set_xlabel("Dia de l'any")

eixos[0].set_ylabel("Temperatura mitjana (°C)")
plt.show()

El grau 2 ja fa una muntanya i baixa l'error a menys de 3 °C. El grau 4 segueix
molt millor la forma de l'onada, amb l'hivern més pla i l'estiu més punxegut, i
s'equivoca uns **2 °C**.

Fins aquí, tot són bones notícies: més grau, millor model. Anem a veure on s'acaba
la festa.

## 5. Quan tenim poques dades

Fins ara hem entrenat amb set anys de dies, més de dues mil cinc-centes mostres.
Imagina ara una estació meteorològica nova, de la qual només tenim **quinze dies
de mesures** repartits per l'any. És una situació molt més habitual del que
sembla: moltes vegades les dades són cares o escasses.

In [ ]:
poques = train.sample(15, random_state=1)

plt.figure(figsize=(8, 4))
plt.scatter(poques["dia_any"], poques["tmed"], s=50, color="black")
plt.xlabel("Dia de l'any")
plt.ylabel("Temperatura mitjana (°C)")
plt.title("Només quinze dies de mesures")
plt.xlim(0, 366)
plt.show()

Quinze punts. Es continua intuint l'onada, però hi ha forats grans, per exemple
entre el març i el juny, on no tenim cap mesura.

Entrenem-hi tres models: un de massa simple, un de raonable i un de molt complex.
Per a cadascun mirarem **dos errors**: el que fa amb aquests mateixos quinze dies i
el que fa amb els dos anys de validació.

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for eix, grau in zip(eixos, [1, 4, 15]):
    fabrica, model = entrenar_polinomi(grau, poques)
    error_train = mae_polinomi(fabrica, model, poques)
    error_valid = mae_polinomi(fabrica, model, valid)

    eix.scatter(valid["dia_any"], valid["tmed"], alpha=0.1, s=6, color="gray", label="validació")
    eix.scatter(poques["dia_any"], poques["tmed"], s=40, color="black", label="els 15 dies")
    eix.plot(dies_grafica, predir_polinomi(fabrica, model, any_sencer), color="tab:red", linewidth=2)
    eix.set_ylim(0, 35)
    eix.set_title(f"Grau {grau}\nentrenament {error_train:.2f} °C · validació {error_valid:.1f} °C")
    eix.set_xlabel("Dia de l'any")

eixos[0].set_ylabel("Temperatura mitjana (°C)")
eixos[0].legend(loc="lower center")
plt.show()

Aquestes tres gràfiques són el centre de tot el notebook. Mira-les amb calma.

**Grau 1.** La recta de sempre. S'equivoca molt amb els quinze dies i molt amb la
validació. Infraajust.

**Grau 4.** La corba passa a prop dels punts sense tocar-los tots i dibuixa una
onada creïble. S'equivoca un grau i mig amb els quinze dies i uns 2,7 °C amb la
validació. És un model raonable.

**Grau 15.** Aquí passa una cosa sorprenent. **La corba passa pràcticament per
sobre dels quinze punts**: l'error d'entrenament és de cinc centèsimes de grau,
gairebé perfecte. I en canvi, entre punt i punt fa **disbarats**: puja i baixa de
manera salvatge i surt del gràfic. L'error de validació passa de 70 °C.

Hem hagut de retallar l'eix vertical perquè la gràfica es pogués llegir. Mira fins
on arriba de veritat:

In [ ]:
fabrica_15, model_15 = entrenar_polinomi(15, poques)
prediccions_15 = predir_polinomi(fabrica_15, model_15, any_sencer)

print(f"Temperatura més baixa que prediu el grau 15: {prediccions_15.min():.0f} °C")
print(f"Temperatura més alta que prediu el grau 15:  {prediccions_15.max():.0f} °C")

Un model que prediu **51 graus sota zero** a finals d'abril i **més de tres mil
graus** l'última setmana de desembre. Són justament els llocs on no hi havia cap
dels quinze punts per retenir la corba. I, alhora, és el model que millor s'ajusta
a les dades d'entrenament.

Això és el **sobreajust** (*overfitting*): el model té tanta llibertat que, en lloc
d'aprendre la forma general de les dades, **les memoritza**, soroll inclòs. Els
quinze punts tenen petites irregularitats de dies concrets que no es tornaran a
repetir, i el grau 15 ha doblegat la corba per passar exactament per totes elles.
Entre punt i punt no hi ha res que el retingui.

És com l'alumne que es prepara un examen memoritzant les respostes del de l'any
passat: ho clava si li posen les mateixes preguntes, i no sap fer res si canvien
una coma.

## 6. Com es detecta el sobreajust sense dibuixar-lo

Amb una sola variable hem pogut dibuixar la corba i veure-hi els disbarats. Però
els models de veritat tenen deu, cent o mil variables, i no es poden dibuixar. Com
sabem, llavors, si un model sobreajusta?

La resposta és a les dues columnes que hem estat mirant: **l'error d'entrenament i
l'error de validació**. Fem-ho per a tots els graus de l'1 al 15.

In [ ]:
escombrada = []
for grau in range(1, 16):
    fabrica, model = entrenar_polinomi(grau, poques)
    escombrada.append({
        "grau": grau,
        "MAE entrenament": mae_polinomi(fabrica, model, poques),
        "MAE validació": mae_polinomi(fabrica, model, valid),
    })

escombrada = pd.DataFrame(escombrada)
escombrada.round(2)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(escombrada["grau"], escombrada["MAE entrenament"], marker="o", label="entrenament (els 15 dies)")
plt.plot(escombrada["grau"], escombrada["MAE validació"], marker="o", label="validació (2022-2023)")
plt.ylim(0, 10)
plt.xticks(range(1, 16))
plt.xlabel("Grau del polinomi")
plt.ylabel("MAE (°C)")
plt.legend()
plt.title("L'error d'entrenament sempre baixa; el de validació, no")
plt.show()

Aquesta gràfica, amb aquesta forma, la tornaràs a veure moltes vegades durant el
curs. Fixa-t'hi bé.

**La línia d'entrenament només baixa.** Cada grau més dóna més llibertat a la corba
per acostar-se als quinze punts, i per tant l'error d'entrenament sempre millora.
Si només miréssim aquesta línia, triaríem el grau 15 i ens equivocaríem de ple.

**La línia de validació fa una vall.** Al principi baixa, perquè el model passa de
massa simple a raonable. Arriba a un mínim, aquí al grau 4. I a partir d'aquí
torna a pujar, primer poc i després de manera desbocada: el model ha començat a
memoritzar. (Hem retallat l'eix a 10 °C; els últims graus se'n van a desenes de
graus.)

D'aquí surt la regla per detectar-ho:

| Què veus | Diagnòstic |
|---|---|
| Error alt a l'entrenament **i** a la validació | **Infraajust**: el model és massa simple |
| Error baix a l'entrenament **i** a la validació, semblants | Model raonable |
| Error baix a l'entrenament però **molt més alt** a la validació | **Sobreajust**: el model memoritza |

**El senyal d'alarma és la distància entre les dues línies.** Recorda el NB 2.2: la
regressió lineal amb deu variables tenia errors d'entrenament i de validació
gairebé iguals. Ara ja saps per què això era una bona notícia.

I aquest és el motiu de fons pel qual existeix el conjunt de validació: **per
triar els hiperparàmetres**, com el grau, sense fer trampa amb el test.

## 7. I si tinguéssim més dades?

Tornem al grau 15, el model desbocat. Què passa si, en lloc de quinze dies, li
donem els més de dos mil cinc-cents dies d'entrenament?

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for eix, (nom, taula) in zip(eixos, [("15 dies", poques), ("tots els dies de 2015-2021", train)]):
    fabrica, model = entrenar_polinomi(15, taula)
    eix.scatter(taula["dia_any"], taula["tmed"], alpha=0.3 if len(taula) > 100 else 1,
                s=8 if len(taula) > 100 else 40, color="black")
    eix.plot(dies_grafica, predir_polinomi(fabrica, model, any_sencer), color="tab:red", linewidth=2)
    eix.set_ylim(0, 35)
    eix.set_title(f"Grau 15 amb {nom}\nentrenament {mae_polinomi(fabrica, model, taula):.2f} °C · "
                  f"validació {mae_polinomi(fabrica, model, valid):.2f} °C")
    eix.set_xlabel("Dia de l'any")

eixos[0].set_ylabel("Temperatura mitjana (°C)")
plt.show()

**El mateix grau 15, amb moltes dades, es comporta bé.** La corba és suau, segueix
l'onada i l'error de validació baixa per sota de 2 °C, fins i tot una mica millor
que el grau 4. Els errors d'entrenament i de validació tornen a ser semblants.

Què ha passat? Amb milers de punts repartits per tot l'any, **no hi ha forats on
la corba es pugui desbocar**, i cap punt sol no la pot estirar cap a ell perquè
n'hi ha molts altres al voltant que la retenen.

La conclusió és important: **el sobreajust no depèn només del model, depèn de la
relació entre la complexitat del model i la quantitat de dades.** Un model massa
complex per a quinze dades pot ser perfectament raonable per a dues mil.

Quan un model sobreajusta, doncs, hi ha dues sortides:

- **Fer el model més simple** (aquí, baixar el grau).
- **Aconseguir més dades**, quan es pot.

Veuràs aquesta mateixa història amb altres models. A la **UT5**, el grau es dirà
*profunditat de l'arbre*. A la **UT6**, el paràmetre `C` de les SVM. A la **UT7**
aprendràs a triar aquests hiperparàmetres d'una manera molt més fiable que amb
una sola partició de validació.

## 8. Resum

- Una regressió lineal pot dibuixar corbes si li fabriquem columnes noves amb
  `PolynomialFeatures`. El **grau** és un hiperparàmetre que decideix quants
  revolts pot fer la corba.
- **Infraajust**: el model és massa simple i s'equivoca molt fins i tot amb les
  dades d'entrenament.
- **Sobreajust**: el model memoritza les dades d'entrenament i s'equivoca molt
  amb les noves.
- Es detecten comparant l'**error d'entrenament** amb l'**error de validació**.
  Una distància gran entre tots dos és el senyal d'alarma.
- L'error d'entrenament **no serveix per triar el model**: sempre prefereix el més
  complex.
- El sobreajust depèn de la complexitat del model **i** de la quantitat de dades.

## 9. Exercicis

**1. Uns altres quinze dies.** Repeteix l'escombrada de la secció 6 canviant
`random_state=1` per `0`, `3` i `7` a l'hora de triar els quinze dies. El millor
grau és sempre el 4? Què et diu això sobre la confiança que podem tenir en una
tria feta amb tan poques dades?

**2. Quaranta dies.** Repeteix la secció 6 amb 40 dies en lloc de 15
(`train.sample(40, random_state=1)`). On és ara el mínim de la validació? El
grau 15 és tan desastrós com abans?

**3. La primavera.** Amb els quinze dies de la secció 5, fes que el model de grau
4 i el de grau 15 prediguin la temperatura del **30 d'abril** (dia 120). Compara-ho
amb la temperatura mitjana real d'aquell dia als anys d'entrenament. Per què el
grau 15 falla tant precisament en aquesta època de l'any? Mira la gràfica dels
quinze punts.

**4. La màxima en lloc de la mitjana.** Repeteix la secció 4.1 fent servir `tmax`
en lloc de `tmed`. Quin grau et sembla suficient? Els errors són més grans o més
petits que amb la mitjana? Per què creus que passa?
Pista: hauràs d'adaptar les funcions, que tenen `"tmed"` escrit a dins.

**5. Per escrit.** Explica amb les teves paraules, en un paràgraf, la diferència
entre un model que *aprèn* i un model que *memoritza*. Fes servir com a exemple
alguna situació de fora de la informàtica que no sigui la de l'examen.

## 10. Per al debat de classe

- Un company us ensenya un model amb un R2 de 0,99 sobre les dades d'entrenament.
  Quina és la primera pregunta que li faríeu?
- Penseu en situacions reals on **només hi ha poques dades** i on, per tant, el
  risc de sobreajust és alt: un producte que fa un mes que es ven, una malaltia
  rara, una botiga que acaba d'obrir... Què faríeu en cada cas?
- Sempre és possible *aconseguir més dades*? Quins costos té?